# Cascade netem test harness

Runs the 4 validation stages for the `--cascade-iface/--cascade-delay-ms/--cascade-jitter-ms/--cascade-loss-pct` patch to `bob_cascade_driver.py` / `alice_cascade_responder.py`, in order:

1. **Regression** — delay off, confirm output shape unchanged.
2. **Loopback smoke test** — arm/disarm mechanics only, not a timing signal (netem on `lo` is unreliable across kernels).
3. **Forced-failure disarm check** — confirm the `finally` fires and nothing is left armed even on a non-convergent Cascade run.
4. **Real FABRIC pair** — single delay value vs baseline, on actual Alice/Bob nodes, checking `reconciliation_elapsed_seconds` moves and the rest of the pipeline doesn't.

Stop and fix before moving to the next stage if a stage fails — don't run the full sweep (that's a separate, later notebook) until Stage 4 shows the expected pattern on one pair.

**Fill in CONFIG below before running.** Paths/hosts/ifaces are placeholders — I don't have your real key-pair file locations or node interface names.

In [18]:
df.upload_project(slice_obj)


=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)


In [19]:
import json
import subprocess
import threading
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # assumes this notebook lives in notebooks/

CONFIG = {
    # A small/real key-pair JSON pair for local stages 1-3. Reuse one of your
    # existing Mock (n=50) or real key-pair files here -- anything small is
    # fine, these stages test the harness, not the science.
    "bob_key_json":   REPO_ROOT / "results" / "bob_sifted_bits_key0.json",
    "alice_key_json": REPO_ROOT / "results" / "alice_sifted_bits_key0.json",
    "qber": 0.01282051282051282,   # REPLACE with the real PE-sample QBER for this key pair
    "k": 390,        # REPLACE with the real PE sample size (m = n_bits + k)
    "port": 5200,
    "local_host": "127.0.0.1",
}

print(f"REPO_ROOT = {REPO_ROOT}")
for k in ("bob_key_json", "alice_key_json"):
    p = Path(CONFIG[k])
    print(f"{k}: {p}  exists={p.exists()}")

REPO_ROOT = /home/fabric/work/qkd-dependability
bob_key_json: /home/fabric/work/qkd-dependability/results/bob_sifted_bits_key0.json  exists=True
alice_key_json: /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key0.json  exists=True


In [20]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as df
fablib = df.get_fablib()

SLICE_NAME = 'qfabric-bb84-2'  # matches 01_setup_slice.ipynb -- don't reprovision, just reconnect
slice_obj = fablib.get_slice(name=SLICE_NAME)
print(f"Reconnected to slice '{SLICE_NAME}' (state: {slice_obj.get_state()})")

alice_node = slice_obj.get_node("alice")
bob_node = slice_obj.get_node("bob")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
Reconnected to slice 'qfabric-bb84-2' (state: StableOK)


In [21]:
def run_local_pair(label, extra_bob_args="", extra_alice_args="", timeout=120):
    """Launch Bob (background thread) then Alice (blocking) locally over
    127.0.0.1, using --no-unique-suffix so output filenames are predictable.
    Returns (bob_result_dict, alice_result_dict). Raises if either process
    exits nonzero or produces no output file -- fail loud here, this is a
    test harness, a silently-empty result should never be treated as a pass.
    """
    bob_out = REPO_ROOT / "results" / f"test_bob_{label}.json"
    alice_out = REPO_ROOT / "results" / f"test_alice_{label}.json"
    for p in (bob_out, alice_out):
        p.parent.mkdir(parents=True, exist_ok=True)
        p.unlink(missing_ok=True)

    bob_cmd = (
        f"python3 {REPO_ROOT}/scripts/bob_cascade_driver.py "
        f"--key-json {CONFIG['bob_key_json']} "
        f"--alice-key-json {CONFIG['alice_key_json']} "
        f"--host 0.0.0.0 --port {CONFIG['port']} "
        f"--qber {CONFIG['qber']} --k {CONFIG['k']} "
        f"--output {bob_out} --no-unique-suffix {extra_bob_args}"
    )
    alice_cmd = (
        f"python3 {REPO_ROOT}/scripts/alice_cascade_responder.py "
        f"--key-json {CONFIG['alice_key_json']} "
        f"--bob-host {CONFIG['local_host']} --port {CONFIG['port']} "
        f"--output {alice_out} --no-unique-suffix {extra_alice_args}"
    )

    bob_proc_holder = {}
    def _start_bob():
        bob_proc_holder["proc"] = subprocess.run(
            bob_cmd, shell=True, capture_output=True, text=True, timeout=timeout,
        )
    bob_thread = threading.Thread(target=_start_bob)
    bob_thread.start()
    time.sleep(2)  # let Bob's ClassicalServer bind before Alice connects

    alice_proc = subprocess.run(alice_cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    bob_thread.join(timeout=timeout)
    bob_proc = bob_proc_holder.get("proc")

    print(f"--- [{label}] bob stdout/stderr (tail) ---")
    print((bob_proc.stdout + bob_proc.stderr)[-1500:] if bob_proc else "BOB DID NOT COMPLETE")
    print(f"--- [{label}] alice stdout/stderr (tail) ---")
    print((alice_proc.stdout + alice_proc.stderr)[-1500:])

    if not bob_out.exists():
        raise RuntimeError(f"[{label}] Bob produced no output file -- check stderr above")
    bob_result = json.loads(bob_out.read_text())
    alice_result = json.loads(alice_out.read_text()) if alice_out.exists() else None
    return bob_result, alice_result

## Stage 1: regression check, delay off
No `--cascade-*` flags. Confirms the patch didn't change behavior when netem is disabled.

In [23]:
import subprocess
print(subprocess.run('ss -ltnp | grep 5200 || true', shell=True, capture_output=True, text=True).stdout)
subprocess.run('pkill -9 -f bob_cascade_driver.py', shell=True)
subprocess.run('pkill -9 -f alice_cascade_responder.py', shell=True)

CompletedProcess(args='pkill -9 -f alice_cascade_responder.py', returncode=-9)

In [22]:
bob_r, alice_r = run_local_pair("stage1")

assert bob_r["cascade_netem"] == {}, f"expected empty cascade_netem, got {bob_r['cascade_netem']}"
assert bob_r["reconciliation_elapsed_seconds"] is not None and bob_r["reconciliation_elapsed_seconds"] > 0
print("STAGE 1 PASS")
print(f"  reconciliation_elapsed_seconds = {bob_r['reconciliation_elapsed_seconds']:.4f}")
print(f"  total elapsed_seconds          = {bob_r['elapsed_seconds']:.4f}")
print(f"  non_convergent = {bob_r['non_convergent']}")

TimeoutExpired: Command 'python3 /home/fabric/work/qkd-dependability/scripts/alice_cascade_responder.py --key-json /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key0.json --bob-host 127.0.0.1 --port 5200 --output /home/fabric/work/qkd-dependability/results/test_alice_stage1.json --no-unique-suffix ' timed out after 120 seconds

## Stage 2: loopback smoke test (mechanics only, NOT a timing signal)
Requires passwordless `sudo tc` on this machine. `--cascade-iface lo` — remember, `netem` on `lo` is unreliable across kernels, so this only checks that arm/disarm run cleanly and bracket the reconcile() call correctly, not that the delay magnitude is trustworthy.

In [5]:
df.upload_project(slice_obj)


=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)


In [8]:
test_node = slice_obj.get_node("alice")  # any one real node works for this

check_cmd = (
    "cd ~/qfabric && ~/qfabric/.venv/bin/python3 -c \""
    "from qne.netem import arm_cascade_netem, disarm_cascade_netem; "
    "import json; "
    "print(json.dumps(arm_cascade_netem('lo', 5200, delay_ms=20))); "
    "print(json.dumps(disarm_cascade_netem('lo')))"
    "\""
)
stdout, stderr = test_node.execute(check_cmd, quiet=True)
print(stdout)
print(stderr)

{"armed_at": 1788980758.7657988, "iface": "lo", "port": 5200, "netem_spec": "netem delay 20ms", "verify_qdisc_show": "qdisc prio 1: root refcnt 2 bands 3 priomap 1 2 2 2 1 2 0 0 1 1 1 1 1 1 1 1\nqdisc netem 30: parent 1:3 limit 1000 delay 20ms"}
{"disarmed_at": 1788980758.7766647, "iface": "lo", "verify_qdisc_show": "qdisc noqueue 0: root refcnt 2"}




In [9]:
check_cmd_real_iface = (
    f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 -c \""
    f"from qne.netem import arm_cascade_netem, disarm_cascade_netem; "
    f"import json; "
    f"print(json.dumps(arm_cascade_netem('{ALICE_IFACE}', 5200, delay_ms=20))); "
    f"print(json.dumps(disarm_cascade_netem('{ALICE_IFACE}')))"
    f"\""
)
stdout, stderr = alice_node.execute(check_cmd_real_iface, quiet=True)
print(stdout)
print(stderr)

NameError: name 'ALICE_IFACE' is not defined

## Stage 3: forced failure — confirm disarm survives a non-convergent run
Push `--reconciliation-prob` high enough to force non-convergence (tune to whatever probability reliably triggers it for your fault injector). Confirm `cascade_netem["disarm"]` is still present, and that `tc qdisc show dev lo` shows nothing left behind afterward.

In [15]:
print(bob_r3["error"])

NameError: name 'bob_r3' is not defined

In [16]:
print(bob_r3["cascade_netem"])

NameError: name 'bob_r3' is not defined

In [17]:
bob_r3, alice_r3 = run_local_pair(
    "stage3",
    extra_bob_args="--cascade-iface lo --cascade-delay-ms 20 --reconciliation-prob 0.9",
    extra_alice_args="--cascade-iface lo --cascade-delay-ms 20",
)



KeyboardInterrupt: 

In [ ]:
print(f"non_convergent = {bob_r3['non_convergent']}")
netem3 = bob_r3["cascade_netem"]
assert "disarm" in netem3, "disarm missing -- the finally did NOT fire on failure, fix before running anything real"
print("disarm present:", netem3["disarm"])

check = subprocess.run("sudo tc qdisc show dev lo", shell=True, capture_output=True, text=True)
print("--- post-run tc qdisc show dev lo ---")
print(check.stdout or "(empty -- good, nothing left armed)")
assert "netem" not in check.stdout, "netem qdisc still present on lo after the run -- LEAK, do not proceed"
print("STAGE 3 PASS")

## Stage 4: real FABRIC pair, single delay value vs baseline
Adapted from the `execute_thread` pattern in `scripts/deploy_fabric.py` (Bob backgrounded, Alice foreground, join). Assumes `slice_obj`, `alice_node`, `bob_node` are already defined earlier in your session (as in `01_setup_slice.ipynb`). **Fill in the real interface names and remote key-pair paths** — `ip a` on each node for the iface, do not reuse `lo` here.

In [ ]:
# --- fill these in ---
alice_node = slice_obj.get_node("alice")
bob_node = slice_obj.get_node("bob")

ALICE_IFACE = alice_node.get_interface(network_name="net_alice_switch").get_device_name()
BOB_IFACE = bob_node.get_interface(network_name="net_switch_bob").get_device_name()

print(f"ALICE_IFACE = {ALICE_IFACE}")
print(f"BOB_IFACE = {BOB_IFACE}")

REMOTE_BOB_KEY_JSON = "~/qfabric/results/bob_sifted_bits_key0.json"
REMOTE_ALICE_KEY_JSON = "~/qfabric/results/alice_sifted_bits_key0.json"
REMOTE_QBER = 0.01282051282051282
REMOTE_K = 390
PORT = 5200
DELAY_CONDITIONS = [0, 50]      # baseline first, then one real delay value

def run_fabric_pair(delay_ms, alice_node, bob_node, bob_classical_ip):
    cascade_flags_bob = f"--cascade-iface {BOB_IFACE} --cascade-delay-ms {delay_ms}" if delay_ms else ""
    cascade_flags_alice = f"--cascade-iface {ALICE_IFACE} --cascade-delay-ms {delay_ms}" if delay_ms else ""

    bob_node.execute("sudo pkill -9 -f bob_cascade_driver 2>/dev/null; sleep 1", quiet=True)
    alice_node.execute("sudo pkill -9 -f alice_cascade_responder 2>/dev/null; sleep 1", quiet=True)

    bob_thread = bob_node.execute_thread(
        f"cd ~/qfabric && sudo -E ~/qfabric/.venv/bin/python3 scripts/bob_cascade_driver.py "
        f"--key-json {REMOTE_BOB_KEY_JSON} --alice-key-json {REMOTE_ALICE_KEY_JSON} "
        f"--host 0.0.0.0 --port {PORT} --qber {REMOTE_QBER} --k {REMOTE_K} "
        f"--output results/test_bob_delay{delay_ms}ms.json --no-unique-suffix {cascade_flags_bob} "
        f"2>&1 | tee /tmp/bob_netem_test.log"
    )
    time.sleep(5)

    alice_thread = alice_node.execute_thread(
        f"cd ~/qfabric && sudo -E ~/qfabric/.venv/bin/python3 scripts/alice_cascade_responder.py "
        f"--key-json {REMOTE_ALICE_KEY_JSON} --bob-host {bob_classical_ip} --port {PORT} "
        f"--output results/test_alice_delay{delay_ms}ms.json --no-unique-suffix {cascade_flags_alice} "
        f"2>&1 | tee /tmp/alice_netem_test.log"
    )

    alice_out, alice_err = alice_thread.result(timeout=180)
    bob_out, bob_err = bob_thread.result(timeout=180)
    print(f"--- delay={delay_ms}ms bob tail ---\n{bob_out[-800:]}")
    print(f"--- delay={delay_ms}ms alice tail ---\n{alice_out[-800:]}")

    stdout, _ = bob_node.execute(f"cat results/test_bob_delay{delay_ms}ms.json", quiet=True)
    return json.loads(stdout)

# results_by_delay = {d: run_fabric_pair(d, alice_node, bob_node, bob_node.get_management_ip()) for d in DELAY_CONDITIONS}
print("Fill in ALICE_IFACE/BOB_IFACE/REMOTE_*_KEY_JSON above, then uncomment the line above to run.")

In [ ]:
# Run this after results_by_delay is populated above.
# The isolation check: reconciliation_elapsed_seconds should move with delay_ms,
# while (elapsed_seconds - reconciliation_elapsed_seconds) -- the PA/verify portion --
# should stay roughly flat across conditions. If PA/verify time also moves with
# delay_ms, the impairment is leaking outside the Cascade phase and this needs
# to be fixed before any real sweep.

# for d, r in results_by_delay.items():
#     recon = r["reconciliation_elapsed_seconds"]
#     other = r["elapsed_seconds"] - recon
#     print(f"delay={d:>4}ms  reconciliation={recon:.3f}s  rest_of_pipeline={other:.3f}s  netem={r['cascade_netem']}")

## Next
Only after Stage 4 shows the expected pattern (reconciliation time moves with delay, rest of pipeline doesn't) on a single real pair: move to the full sweep (multiple delay/jitter/loss values x multiple key pairs x replicates) in a separate notebook, reusing `sweep_utils.py` patterns rather than this harness.